# 🧪 Virtual Screening with CADD

## 🔍 Screening Criteria

We will evaluate compounds using the following key filters:

- **Lipinski's Rule of Five** — To assess oral bioavailability.
- **Quantitative Estimate of Drug-Likeness (QED)** — A score that combines multiple properties into a single drug-likeness metric.
- **Synthetic Accessibility Score (SAscore)** — To estimate the ease of chemical synthesis.
- **PAINS (Pan Assay Interference Compounds)** — To flag problematic substructures that may lead to false positives in screening assays.
- **Tanimoto Similarity** — To measure molecular similarity (e.g., with known active compounds) based on fingerprints.

Assignment 
1. ROV = 0
2. QED = >= 0.67
3. SAScore = <=4
4. PAINS
5. Tanimoto: ref Smiles = "C1=CC2=C(C=CC(=C2))N=C1", 0.3


In [ ]:
import rdkit

In [ ]:
!pip install rdkit

In [ ]:
!pip install pandas

In [ ]:
import pandas as pd
import rdkit 
from rdkit import Chem

In [ ]:
df = pd.read_csv("chembl_22_clean_1576904_sorted_std_final.smi", sep="\t", header=None, names=["smiles", "name"])

In [ ]:
df

In [ ]:
df = df.sample(100000, random_state=2025)

In [ ]:
df

In [ ]:
df.reset_index(inplace=True, drop=True)

In [ ]:
df

In [ ]:
df.isna().sum()

## 💊 Lipinski's Rule of Five

Lipinski's Rule of Five is a widely used guideline in drug discovery to evaluate the **oral bioavailability** of a compound. It helps identify molecules that are more likely to become orally active drugs in humans.

A compound is considered likely to have good oral bioavailability if **no more than one** of the following rules is violated:

- 🔹 **Molecular weight** ≤ 500 Da  
- 🔹 **LogP** (octanol-water partition coefficient) ≤ 5  
- 🔹 **Hydrogen bond donors** (OH and NH groups) ≤ 5  
- 🔹 **Hydrogen bond acceptors** (N and O atoms) ≤ 10  

These rules are not absolute but serve as a **filter to prioritize drug-like molecules** during virtual screening.

> 📌 Note: Compounds that violate multiple rules may still be biologically active but are less likely to be orally bioavailable.



In [ ]:
df

In [ ]:
df['Mol'] = df['smiles'].apply(Chem.MolFromSmiles)

In [ ]:
df

In [ ]:
df['Mol'][11]

In [ ]:
from rdkit.Chem import Descriptors
from rdkit.Chem.Descriptors import rdMolDescriptors



In [ ]:
df.isna().sum()

In [ ]:
df.dropna(inplace=True)
df.reset_index(inplace=True, drop=True)

In [ ]:
df

In [ ]:
df['MolWt'] = df['Mol'].apply(Descriptors.MolWt)

In [ ]:
df

In [ ]:
df['LogP'] = df['Mol'].apply(Descriptors.MolLogP)

In [ ]:
df['HbondDonors'] = df['Mol'].apply(Descriptors.NumHDonors)

In [ ]:
df['HbondAcceptors'] = df['Mol'].apply(Descriptors.NumHAcceptors)

In [ ]:
df

In [ ]:
df_filtered = df[
    (df['MolWt'] <= 500) &
    #(df['MolWt'] >= 200) &
    (df['LogP'] <= 5) &
    (df['HbondDonors'] <= 5) &
    (df['HbondAcceptors'] <= 10)
].copy()


In [ ]:
df_filtered.reset_index(inplace=True, drop=True)

In [ ]:
df_filtered

In [ ]:
import matplotlib

In [ ]:
import matplotlib as plt

In [ ]:
df_filtered['MolWt'].hist(figsize=(8, 5), edgecolor='black', color= 'magenta')

## ⚖️ QED (Quantitative Estimate of Drug-likeness)

QED is a metric that quantifies how "drug-like" a molecule is, based on a combination of several important physicochemical properties. It outputs a **single score between 0 and 1**, where **higher values indicate better drug-likeness** [(Bickerton et al., 2022)](https://pmc.ncbi.nlm.nih.gov/articles/PMC3524573/#:~:text=To%20quantify%20compound%20quality%20we,one%20(all%20properties%20favourable).).

### 🧬 Properties Considered in QED
QED combines the following molecular features:

- Molecular weight (MW)
- LogP (lipophilicity)
- Hydrogen bond donors (HBD)
- Hydrogen bond acceptors (HBA)
- Rotatable bonds
- Aromatic ring count
- Structural alerts
- Topological polar surface area (TPSA)

### QED values can range between zero (all properties unfavourable) and one (all properties favourable).

### 📊 QED Score Range and Interpretation

| QED Score | Interpretation             |
|-----------|----------------------------|
| 0.9 – 1.0 | **Highly drug-like**       |
| 0.7 – 0.9 | **Moderately drug-like**   |
| 0.4 – 0.7 | **Potentially drug-like**  |
| < 0.4     | **Poor drug-likeness**     |

Most marketed oral drugs tend to have QED scores **above 0.67**, but this is not a strict cutoff.

> 📌 QED is especially useful for ranking and prioritizing compounds during virtual screening campaigns, as it provides a better view of drug-likeness rather than binary filtering.


In [ ]:
df

In [ ]:
df['Qed'] = df['Mol'].apply(Descriptors.qed)

In [ ]:
df

In [ ]:
df['Qed'].hist(figsize=(8, 5), edgecolor='black', color= 'magenta')

In [ ]:
df_filtered_qed =  df[df['Qed'] >= 0.67]

In [ ]:
df_filtered_qed

In [ ]:
df_filtered_qed.reset_index(inplace=True,drop= True)

In [ ]:
df_filtered_qed['Qed'].hist(figsize=(8, 5), edgecolor='black', color= 'magenta')

In [ ]:
df_filtered_qed

## 🛠️ Synthetic Accessibility (SA) Score

The **Synthetic Accessibility (SA) Score** estimates how easy or difficult it would be to **chemically synthesize a compound** in the laboratory. It is a useful metric in drug discovery, especially after virtual screening, to prioritize molecules that are not only active but also practical to make.

### 🔬 How It Works
The SA score combines two key components:
1. **Fragment contributions** — How common or rare the molecular fragments are based on a large dataset of known synthetic compounds.
2. **Complexity penalties** — Structural features that increase synthetic difficulty, such as:
   - Large ring systems
   - Highly branched structures
   - Chiral centers
   - Unusual functional groups

### 📊 SA Score Range and Interpretation

| SA Score | Interpretation              |
|----------|-----------------------------|
| 1.0 – 3.0 | **Easy to synthesize**      |
| 3.1 – 5.0 | **Moderate difficulty**     |
| 5.1 – 10.0| **Difficult to synthesize** |

> ✅ Lower SA scores are more desirable when selecting compounds for further development, as they indicate a higher likelihood of successful synthesis.

### 🧪 Why It Matters
Even the most promising compound in terms of activity and drug-likeness may not be useful if it can't be synthesized efficiently. SA score helps avoid such dead ends early in the pipeline.


In [ ]:
from rdkit.Contrib.SA_Score import sascorer

In [ ]:
df['sascore'] = df['Mol'].apply(sascorer.calculateScore)

In [ ]:
df

In [ ]:
df['sascore'].hist(bins=20, figsize=(10,5), edgecolor='black', color= 'magenta')

In [ ]:
df_filtered = df[df['sascore'] <= 3.0]

In [ ]:
df_filtered

In [ ]:
df_filtered.reset_index(inplace=True, drop= True)

In [ ]:
df_filtered

## 🚫 PAINS (Pan Assay Interference Compounds)

**PAINS** are **Pan Assay Interference Compounds**—molecules that tend to give **false positive results** in biological assays. These compounds interfere with multiple assay types, not through specific biological activity, but through non-specific or reactive mechanisms.

### ⚠️ Why PAINS Are Problematic

- They may appear as "hits" in virtual screening or high-throughput screening campaigns.
- They often interfere with assay readouts due to:
  - Redox cycling
  - Covalent modification of proteins
  - Aggregation
  - Fluorescence interference

Because of this, they can **mislead drug discovery efforts** by diverting time and resources toward compounds with no true pharmacological potential.

### 🔬 How Are PAINS Identified?

PAINS filters use a set of predefined **substructure patterns** (SMARTS) to flag known problematic motifs. These filters were originally developed by Baell and Holloway (2010) after analyzing common assay artifacts.

### 🧪 In Practice

Before advancing any compound, it's good practice to **screen for PAINS** using computational filters (e.g., using RDKit or other cheminformatics tools) and **remove or deprioritize** flagged molecules.

> ✅ Removing PAINS helps improve the quality and reliability of your virtual screening pipeline.



In [ ]:
from rdkit import Chem
from rdkit.Chem import PandasTools
from rdkit.Chem.FilterCatalog import *
from rdkit.Chem import Descriptors, Draw, PandasTools

In [ ]:
df

In [ ]:
#filter by pains
params = FilterCatalogParams()
params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
catalog = FilterCatalog(params)

In [ ]:
substructures = pd.read_csv("pains_list.csv", sep="\s+")  
substructures["rdkit_molecule"] = substructures.smarts.apply(Chem.MolFromSmarts)
print("Number of unwanted substructures in collection:", len(substructures))

In [ ]:
substructures


In [ ]:
!pip install tqdm

In [ ]:
import tqdm

In [ ]:
!pip install rdkit

In [ ]:
import rdkit

In [ ]:
from tqdm import tqdm

# search for unwanted substructure
matches = []
clean = []
for index, row in tqdm(df.iterrows(), total=df.shape[0]):
    molecule = Chem.MolFromSmiles(row.smiles)
    match = False
    for _, substructure in substructures.iterrows():
        if molecule.HasSubstructMatch(substructure.rdkit_molecule):
            matches.append(
                {
                    "smiles": row.smiles,
                    "rdkit_molecule": molecule,
                    "substructure": substructure.rdkit_molecule,
                    "substructure_name": substructure["name"],
                }
            )
            match = True
    if not match:
        clean.append(index)
        

matches = pd.DataFrame(matches)
df_clean = df.loc[clean]




In [ ]:
df_clean

In [ ]:
to_highlight = [
    row.rdkit_molecule.GetSubstructMatch(row.substructure) for _, row in matches.head(5).iterrows()
]
Chem.Draw.MolsToGridImage(
    list(matches.head(5).rdkit_molecule), molsPerRow=5,
    highlightAtomLists=to_highlight,
    legends=list(matches.head(5).substructure_name),
)



In [ ]:
df

In [ ]:
matches

In [ ]:

from rdkit.Chem import Draw
from collections import defaultdict
import random

pains_groups = defaultdict(list)

for idx, row in matches.iterrows():
    substructure_name = row['substructure_name']
    smiles = row['smiles']
    pains_groups[substructure_name].append(smiles)

# Step 2: Randomly select one molecule from each PAINS substructure group
selected_smiles = {group: random.choice(smiles_list) for group, smiles_list in pains_groups.items()}

# Step 3: Prepare molecules and highlight PAINS substructures
mols = []
highlight_atoms = []
legends = []

# Iterate through each selected molecule for a PAINS substructure
for pains_group, smiles in selected_smiles.items():
    mol = Chem.MolFromSmiles(smiles)
    mols.append(mol)
    
    # Get substructure match for PAINS
    substructure = matches[matches['smiles'] == smiles]['substructure'].values[0]
    match = mol.GetSubstructMatch(substructure)
    
    # Highlight the atoms in the substructure match
    if match:
        highlight_atoms.append(list(match))
    else:
        highlight_atoms.append([])  # If no match found, append empty list
    
    legends.append(f'{pains_group}')  # Add the group name to the legend

# Step 4: Draw molecules with highlighted PAINS substructures
img = Draw.MolsToGridImage(
    mols, 
    molsPerRow=4,  # 4 molecules per row
    subImgSize=(300, 300),  # Size of each subimage
    highlightAtomLists=highlight_atoms,  # Highlight PAINS substructures
    legends=legends  # Labels for the PAINS groups
)

# Step 5: Display the image
img

In [ ]:
df_clean

In [ ]:
matches

#### You will notice the number of compounds in the `Matches` and `df_clean` dataframes do not sum up to our starting `df` number?
# WHY

In [ ]:
matches.drop_duplicates(subset='smiles', inplace=True)

In [ ]:
matches

In [ ]:
matches.reset_index(inplace=True)

In [ ]:
matches

In [ ]:
df_clean

In [ ]:
df_clean.reset_index(inplace=True)

In [ ]:
df_clean

## 🤝 Tanimoto Similarity

**Tanimoto Similarity** is a widely used metric in cheminformatics to compare the **structural similarity** between two molecules, typically using their **molecular fingerprints** (such as ECFP).

### 🧪 What It Measures

Tanimoto similarity quantifies how many structural features two molecules share:

$$
\text{Tanimoto Similarity} = \frac{c}{a + b - c}
$$

Where:  
- \( a \): Number of bits set to 1 in fingerprint A  
- \( b \): Number of bits set to 1 in fingerprint B  
- \( c \): Number of bits set to 1 in both A and B (the intersection)

### 📊 Similarity Score Range

| Tanimoto Score | Interpretation         |
|----------------|------------------------|
| 1.0            | Identical structures    |
| 0.7 – 0.99     | Highly similar          |
| 0.4 – 0.7      | Moderately similar      |
| < 0.4          | Weak or no similarity   |

### 🔍 Why It’s Useful

Tanimoto similarity is commonly used for:
- **Similarity-based virtual screening**
- Identifying analogs of known active compounds
- **Clustering** compounds by structure

> ✅ Molecules with a Tanimoto score above **0.7** to a known active are often prioritized in drug discovery.


In [ ]:
df

In [ ]:
from rdkit.Chem import AllChem
from rdkit import Chem, DataStructs
tqdm.pandas()

df['Fingerprint'] = df['Mol'].apply(lambda mol: AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024))


ref_smiles = 'C1=CC2=C(C=CC(=C2))N=C1'  

ref_mol = Chem.MolFromSmiles(ref_smiles)
ref_fp = AllChem.GetMorganFingerprintAsBitVect(ref_mol, radius=2, nBits=1024)


df['Tanimoto'] = df['Fingerprint'].progress_apply(lambda fp: DataStructs.TanimotoSimilarity(ref_fp, fp))


df_filtered = df[df['Tanimoto'] >= 0.1].copy()

df_filtered

In [ ]:
df_filtered.reset_index(inplace=True, drop=True)

In [ ]:
df_filtered

In [ ]:
df_filtered[df_filtered['Tanimoto'] >= 0.3].shape

In [ ]:
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Draw
from rdkit.Chem.Draw import rdMolDraw2D

# Filter and get top 5 by Tanimoto similarity
top5 = df_filtered.sort_values(by='Tanimoto', ascending=False).head(5)

# Convert SMILES to Mol objects (if needed)
top5_mols = [mol for mol in top5['Mol'] if mol is not None]

# Reference molecule from SMILES
ref_mol = Chem.MolFromSmiles(ref_smiles)
AllChem.Compute2DCoords(ref_mol)

# Generate 2D coordinates and align to reference
for mol in top5_mols:
    AllChem.Compute2DCoords(mol)
    try:
        AllChem.GenerateDepictionMatching2DStructure(mol, ref_mol)
    except ValueError:
        # Fallback: just use standard 2D coords
        pass

# Combine reference with top 5
mols_to_draw = [ref_mol] + top5_mols
legends = ['Reference'] + [f"Tanimoto: {sim:.2f}\nChemblID: {name}" 
                           for sim, name in zip(top5['Tanimoto'], top5['name'])]

# Highlight atoms (safely handle missing substructure matches)
highlight_atom_lists = []
for mol in mols_to_draw:
    if mol.HasSubstructMatch(ref_mol):
        match = mol.GetSubstructMatch(ref_mol)
    else:
        match = ()
    highlight_atom_lists.append(match)

# Draw grid image
img = Draw.MolsToGridImage(
    mols_to_draw,
    molsPerRow=3,
    subImgSize=(400, 300),
    legends=legends,
    highlightAtomLists=highlight_atom_lists
)

img
